# Data11 amplitude analysis

Run all cells from the release environment. IM and accelerometer were acquired separately; reported matching drive, position and load support condition-mean comparisons, not synchronized file pairing. The confirmed accelerometer range was ±16 g; `Raw_Z * 0.000488` is a nominal conversion, not an individual calibration. Clock accuracy, analog bandwidth and batch drift remain unresolved.

The current analysis uses IM_53/55/56 and accel_10/11/12 at 100 Hz, with levels 25, 24, 23, 22, 21, 20. Acquisition blocks are 30 s and both sensors use central 26 s windows. IM_54 is previewed but is not a selected independent repeat. `start_s=0` is a segmentation assumption, not a measured switching time; inspect envelopes and timing diagnostics.

Peak-to-peak, demeaned RMS, four whole-spectrum FFT estimates and the separate 100±1 Hz target peak remain distinct. Save peak frequency alongside amplitude: the global maximum need not be 100 Hz. Volume is an amplifier setting, not a physical input amplitude.

Means and sample SD use independent files; a single file has no SD. Strict reference ratios require accepted gaps and use IM mean peak counts / accelerometer mean peak g. Diagnostic ratios retain gap warnings. Free-intercept descriptive OLS slopes are separate from pointwise counts/g. The fit does not fully model errors in both axes, between-level dependence or systematic calibration uncertainty; it is not a detection limit or sound-pressure sensitivity.


In [ ]:
# data11-cell: setup
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "release_manifest.json").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
from portable_runtime import project_root, data_root, ensure_environment, notebook_path
PROJECT = project_root()
ensure_environment()
NOTEBOOK_PATH = notebook_path("data11_amplitude_analysis")
# Reload amplitude modules so a live kernel uses the current source.
import importlib
import data11_config
import data11_amplitude_plots
import data11_analysis
importlib.invalidate_caches()
importlib.reload(data11_config)
importlib.reload(data11_amplitude_plots)
importlib.reload(data11_analysis)
from data11_config import default_config
from data11_analysis import manifest, run_analysis, METHODS
from data11_analysis import manifest, run_analysis, METHODS

pd.set_option("display.max_columns", 24)
pd.set_option("display.precision", 8)
print("Python:", sys.executable)

# Reload the shared paper figure style for this Run All.
import importlib as _paper_importlib
import paper_figure_style as _paper_style
_paper_importlib.reload(_paper_style)


## Central windows and candidate preview

Both `IM_CENTER_WINDOW_S` and `ACCEL_CENTER_WINDOW_S` are 26 s, retaining 2-28 s of each 30 s level. The preview includes IM_53/54/55/56; the selection below retains three files. Accelerometer IDs 10/11/12 have confirmed 30 s levels; legacy IDs 7/8/9 had 20 s levels. Windows longer than a block are rejected.

The main estimate uses the full central window. Stability uses complete 4 s subwindows: six cover 24 s, with the remaining 2 s reported and excluded only from stability. Subwindows are not independent repeats. Do not move segmentation merely to obtain monotonic responses.


In [ ]:
# data11-cell: candidate_preview
import data11_amplitude_review as amplitude_review
importlib.reload(amplitude_review)
# ===== User controls: retained duration, NOT acquisition dwell =====
IM_CENTER_WINDOW_S = 26.0       # 30 s IM levels: e.g. 26, 24, 20, 16
ACCEL_CENTER_WINDOW_S = 26.0    # accel_10/11/12: 30 s per level, retain 2-28 s
ACCEL_RECORD_IDS = [10, 11, 12] # accel_7/8/9 have 20 s levels; use a window <= 20 s
STABILITY_SUBWINDOW_S = 4.0
AMPLITUDE_CANDIDATE_IDS = [53, 54, 55, 56]  # Confirmed 30 s IM records

AMPLITUDE_PREVIEW_CONFIG = amplitude_review.preview_config()
AMPLITUDE_PREVIEW_CONFIG["data_root"] = str(data_root("Data11"))
AMPLITUDE_PREVIEW_CONFIG["files"]["IM"] = AMPLITUDE_CANDIDATE_IDS.copy()
AMPLITUDE_PREVIEW_CONFIG["central_window_s"] = IM_CENTER_WINDOW_S
AMPLITUDE_PREVIEW_CONFIG["central_window_overrides_s"] = {
    f"accel_{i}.csv": ACCEL_CENTER_WINDOW_S for i in ACCEL_RECORD_IDS
}
AMPLITUDE_PREVIEW_CONFIG["subwindow_s"] = STABILITY_SUBWINDOW_S
AMPLITUDE_PREVIEW = amplitude_review.run_preview(AMPLITUDE_PREVIEW_CONFIG, NOTEBOOK_PATH)
for path in AMPLITUDE_PREVIEW["figures"]:
    display(Image(filename=path))
display(AMPLITUDE_PREVIEW["summary"])
print("All-candidate output:", AMPLITUDE_PREVIEW["output_path"])

# Optional original raw-envelope / 1 s RMS diagnostics (a separate metric).
SHOW_RAW_OVERVIEWS = False
if SHOW_RAW_OVERVIEWS:
    import data11_record_review as review
    importlib.reload(review)
    AMPLITUDE_REVIEW_CONFIG = review.review_config()
    AMPLITUDE_REVIEW_CONFIG['tracking']['materials'] = {}
    AMPLITUDE_REVIEW_CONFIG['include_raw_examples'] = False
    AMPLITUDE_REVIEW = review.run_review(AMPLITUDE_REVIEW_CONFIG, NOTEBOOK_PATH)
    display(AMPLITUDE_REVIEW['audit'])
    print('All-candidate output:', AMPLITUDE_REVIEW['output_path'])
    for path in AMPLITUDE_REVIEW['figures']:
        display(Image(filename=path))


## Configuration

These selections apply to Data11. Save the notebook after editing the settings.


In [ ]:
# data11-cell: configuration
CONFIG = default_config("amplitude")
CONFIG["data_root"] = AMPLITUDE_PREVIEW_CONFIG["data_root"]
CONFIG["accel_g_per_count"] = 0.000488  # Reported accelerometer range: ±16 g
CONFIG["reference_conditions_confirmed"] = True
# Select independent recordings
SELECTED_AMPLITUDE_IDS = [53, 55, 56]  # Choose three distinct IDs from AMPLITUDE_CANDIDATE_IDS.
if len(SELECTED_AMPLITUDE_IDS) != 3 or len(set(SELECTED_AMPLITUDE_IDS)) != 3:
    raise ValueError('Review the candidates and select three distinct IDs in SELECTED_AMPLITUDE_IDS.')
if not set(SELECTED_AMPLITUDE_IDS).issubset(set(AMPLITUDE_CANDIDATE_IDS)):
    raise ValueError('Select IDs from the confirmed amplitude candidates.')
CONFIG["files"] = {
    "IM": SELECTED_AMPLITUDE_IDS,       # IDs or explicit CSV filenames are accepted
    "accel": ACCEL_RECORD_IDS.copy(), # Independent count; [] disables this sensor
}
CONFIG["max_files"] = {
    "IM": None,              # None: all; integer: first N; 0: disabled
    "accel": None,
}
# File lists define the selection; no cross-sensor pairing is implied.
# Missing files are reported without automatic replacement.

# Analysis windows
# Copy the exact schedule/window controls used for the all-candidate preview.
# Edit AMPLITUDE_PREVIEW_CONFIG above and rerun preview to change timing.
from copy import deepcopy
for key in ("levels", "dwell_s", "dwell_overrides_s", "start_s", "start_overrides_s",
            "bounds_overrides_s", "central_window_s", "central_window_overrides_s", "trim_s", "subwindow_s",
            "target_hz", "target_halfwidth_hz", "schedule_source"):
    CONFIG[key] = deepcopy(AMPLITUDE_PREVIEW_CONFIG[key])
from data11_config import amplitude_files
SELECTED_FILES = amplitude_files(CONFIG)
display(pd.DataFrame([{"sensor": sensor, "selected_files": names, "n_files": len(names)}
                      for sensor, names in SELECTED_FILES.items()]))

# Whole-spectrum peak search and post-selection checks
CONFIG["global_search_min_hz"] = 1.0  # Strictly f > 1 Hz, through Nyquist
CONFIG["amplitude_peak_check_halfwidth_hz"] = 1.0  # Check 100±1 Hz after the unrestricted peak search

# Save square peak-response and counts/g PNG/SVG figures.
CONFIG["amplitude_reference_square_figures"] = True
CONFIG["amplitude_reference_figure_size_in"] = 7.0  # Base side in inches; minimum 7; reserve legend space



## Inputs and methods

Check input roles, paths and missing files. This table does not analyze or modify raw CSV files. The run records the full parameters and methods.


In [ ]:
# data11-cell: preview
INPUT_PLAN = pd.DataFrame(manifest(CONFIG))
INPUT_PLAN["exists"] = INPUT_PLAN.path.map(lambda p: Path(p).is_file())
display(INPUT_PLAN)
for key, text in METHODS.items():
    if key in {"input", "timing", "reference", CONFIG["analysis"], CONFIG["analysis"]+"_global_peak"}:
        print(key + ": " + text + "\n")


## Run

Each execution creates a new dated run under the configured results directory. Existing runs and raw data are preserved. Failures retain partial outputs and an explicit status.


In [ ]:
# data11-cell: run
# Invalidate the previous result before attempting a new run.
RESULTS = None
OUTPUT_PATH = None
# Reload amplitude modules so a live kernel uses the current source.
import importlib
import data11_config
import data11_amplitude_plots
import data11_analysis
importlib.invalidate_caches()
importlib.reload(data11_config)
importlib.reload(data11_amplitude_plots)
importlib.reload(data11_analysis)
from data11_config import default_config
from data11_analysis import manifest, run_analysis, METHODS

shell = get_ipython()
history = shell.history_manager.input_hist_raw if shell is not None else []
live_cells = [s for s in history if s.lstrip().startswith("# data11-cell:")]
if not CONFIG.get('files') or CONFIG['files'].get('IM') != SELECTED_AMPLITUDE_IDS or len(SELECTED_AMPLITUDE_IDS) != 3:
    raise RuntimeError('Run the three-record selection cell successfully first.')
RESULTS = run_analysis(CONFIG, NOTEBOOK_PATH, executed_cells=live_cells)
OUTPUT_PATH = Path(RESULTS["output_path"])
print("Completed:", OUTPUT_PATH)


## Results

Sample SD describes variation between independent recordings. Interpret estimates alongside the quality audit and actual supported windows.


## Whole-spectrum peak checks

The added response uses the largest FFT bin at strictly f > 1 Hz up to Nyquist, with 8x zero padding. It does not constrain the search to 100 Hz or replace an unexpected peak. The separate target-band result uses interpolation within 100±1 Hz.

`amplitude_peak_check_halfwidth_hz` checks the selected global peak afterward. If any selected recording fails that check at a level, a common-frequency counts/g ratio and OLS point are not calculated for that level. All peaks remain in plots and CSVs. Error bars are sample SD across files; 4 s stability windows do not enter the main response.


In [ ]:
# data11-cell: tables
# Require results from the updated amplitude run cell.
_required_results = {"global_peak_check", "global_peak_reference", "global_peak_fit"}
if not isinstance(globals().get("RESULTS"), dict) or not _required_results.issubset(RESULTS):
    raise RuntimeError(
        "RESULTS are missing or incomplete. Run the analysis cell above first. "
        "Wait for Completed before displaying results; execution creates a new run."
    )

print("summary")
display(RESULTS["summary"])
print("reference")
display(RESULTS["reference"])
print("fit")
display(RESULTS["fit"])

print("global_peak_check")
display(RESULTS["global_peak_check"])
print("global_peak_reference")
display(RESULTS["global_peak_reference"])
print("global_peak_fit")
display(RESULTS["global_peak_fit"])


In [ ]:
# data11-cell: figures
# Require results from the updated amplitude run cell.
_required_results = {"global_peak_check", "global_peak_reference", "global_peak_fit"}
if not isinstance(globals().get("RESULTS"), dict) or not _required_results.issubset(RESULTS):
    raise RuntimeError(
        "RESULTS are missing or incomplete. Run the analysis cell above first. "
        "Wait for Completed before displaying results; execution creates a new run."
    )

def show_figures(*keywords):
    for path in RESULTS["figures"]:
        if any(word in Path(path).stem for word in keywords):
            display(Image(filename=path))
show_figures("amplitude_global_peak")
show_figures("amplitude_metrics", "amplitude_reference_response", "amplitude_stability")

## Quality and output files

`dt_cv` describes sampling-interval variation. A recording-level gap may lie outside an analyzed window; inspect window-level `gap_ok` as well. All per-recording diagnostics are saved in the output directory.


In [ ]:
# data11-cell: diagnostics
# Require results from the updated amplitude run cell.
_required_results = {"global_peak_check", "global_peak_reference", "global_peak_fit"}
if not isinstance(globals().get("RESULTS"), dict) or not _required_results.issubset(RESULTS):
    raise RuntimeError(
        "RESULTS are missing or incomplete. Run the analysis cell above first. "
        "Wait for Completed before displaying results; execution creates a new run."
    )

display(RESULTS["audit"])
display(pd.DataFrame(RESULTS["input_issues"]))
print("Exact output path:", OUTPUT_PATH)
print("PNG / CSV / parameters.json / inputs.json / provenance.json / methods.txt / run_status.json / scripts/")
print("Limitations: uncalibrated timebase; nominal accelerometer conversion; independent batches; inspect excluded windows and records.")


## Method references

Data11 uses its own parameters and functions. Related Data10 notebooks retain their separate metric definitions. Implementation references: [SciPy Hann window](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.windows.hann.html) and [SciPy spectrogram](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.spectrogram.html).
